# Salinity

In [21]:
from pathlib import Path
import re
import shutil
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
import numpy as np
import rasterio
from rasterio.transform import from_origin
from rasterio.shutil import copy as rio_copy

def process_xyz_to_geospatial(file_path):
    # Read the xyz file into a pandas DataFrame
    df = pd.read_csv(file_path, delim_whitespace=True, header=None, names=['x', 'y', 'z'])

    # Create geometry column (2D point)
    df['geometry'] = df.apply(lambda row: Point(row['x'], row['y']), axis=1)

    # Convert to GeoDataFrame and set CRS to WGS-84 / UTM 48N
    gdf = gpd.GeoDataFrame(df, geometry='geometry', crs="EPSG:32648")

    # --- Create raster before dropping NaNs ---
    df_raster = df.dropna(subset=['z'])

    # Define raster resolution
    resolution = 2000  # meters

    # Compute bounds and expand them by half a cell so points are in the middle
    xmin, ymin, xmax, ymax = df_raster['x'].min(), df_raster['y'].min(), df_raster['x'].max(), df_raster['y'].max()
    xmin -= resolution / 2
    ymin -= resolution / 2
    xmax += resolution / 2
    ymax += resolution / 2

    # Define raster size
    width = int(np.ceil((xmax - xmin) / resolution))
    height = int(np.ceil((ymax - ymin) / resolution))

    # Adjust xmax/ymax based on integer width/height to align grid properly
    xmax = xmin + width * resolution
    ymax = ymin + height * resolution

    # Create affine transform
    transform = from_origin(xmin, ymax, resolution, resolution)

    # Create empty raster array
    raster = np.full((height, width), np.nan)

    # Map points to raster cells (point assigned to cell containing its center)
    for _, row in df_raster.iterrows():
        col = int((row['x'] - xmin) // resolution)
        row_idx = int((ymax - row['y']) // resolution)
        if 0 <= row_idx < height and 0 <= col < width:
            raster[row_idx, col] = row['z']

    # Save initial GeoTIFF
    temp_tif = file_path.replace('.xyz', '_temp.tif')
    with rasterio.open(
        temp_tif,
        'w',
        driver='GTiff',
        height=height,
        width=width,
        count=1,
        dtype=raster.dtype,
        crs="EPSG:32648",
        transform=transform,
        nodata=np.nan,
        compress='LZW'
    ) as dst:
        dst.write(raster, 1)

    # Extract the number at the end and convert to full year
    match = re.search(r"_(cc\d{2}[a-z0-9]+?)(\d+)\.xyz$", Path(file_path).name.lower())
    if match:
        number_part = match.group(2)       # e.g., "18"
        year = f"20{number_part}"          # convert to "2018"

        # New file path with only the year as the filename
        cog_tif = Path(file_path).with_name(f"{year}.tif")

    rio_copy(temp_tif, cog_tif, copy_src_overviews=True, driver='COG', compress='LZW')

    # Remove temporary file
    Path(temp_tif).unlink() 

    # Remove .xyz file
    Path(file_path).unlink()

    # Remove any .xml in the folder
    for xml_file in Path(file_path).parent.glob("*.xml"):
        xml_file.unlink()

    print(f"Cloud Optimized GeoTIFF saved to {cog_tif}")

    # --- Continue with GeoJSON ---
    gdf.dropna(inplace=True)
    output_path = file_path.replace('.xyz', '.geojson')
    gdf.to_file(output_path, driver='GeoJSON')

    print(f"GeoJSON saved to {output_path}")

    return cog_tif

# Base folder
base_folder = Path(r"N:\Deltabox\Postbox\Athanasiou, Panos\van_Sepehr\salinity_mekong\projections_gridded")

# List to store copied file paths
copied_files = []

# List of allowed suffixes
allowed_suffixes = ["y", "sb2y", "sm2y", "sb2rb1y", "sm2rb1y"]

# Create output folders and copy files based on pattern
for file in base_folder.glob("*.xyz"):
    fname = file.name.lower()

    # Extract pattern: starts with "_cc" followed by digits and letters, before the final number
    match = re.search(r"_(cc\d{2}[a-z0-9]+?)(\d+)\.xyz$", fname)
    if not match:
        print(f"⚠️ Skipping file (no match found): {fname}")
        continue

    pattern_main = match.group(1)  # e.g., cc45blsm2rb1y
    number_part = match.group(2)   # e.g., 40

    # Determine the suffix after the first digit(s) in ccXX...
    # Remove the initial ccXX
    suffix = pattern_main[4:]  # skip "cc" + two digits

    if suffix not in allowed_suffixes:
        print(f"⚠️ Skipping file (suffix not allowed): {fname}")
        continue

    # Create target folder
    target_folder = base_folder / pattern_main
    target_folder.mkdir(exist_ok=True)

    # Copy the file
    target_path = target_folder / file.name
    shutil.copy2(file, target_path)
    copied_files.append(str(target_path))
    print(f"✅ Copied: {file.name} → {pattern_main}/")

print("\n🎉 Done organizing files!")
print(f"\n📄 List of copied files ({len(copied_files)}):")
for f in copied_files:
    print(f)

cog_files = []

for file_path in copied_files[:1]:
    cog_file = process_xyz_to_geospatial(file_path)
    cog_files.append(cog_file)

✅ Copied: P50_cc45sm2rb1y30.xyz → cc45sm2rb1y/
✅ Copied: P50_cc45y18.xyz → cc45y/
✅ Copied: P50_cc45sm2y50.xyz → cc45sm2y/
✅ Copied: P50_cc85y40.xyz → cc85y/
✅ Copied: P50_cc45y40.xyz → cc45y/
⚠️ Skipping file (suffix not allowed): p50_cc45qd10sm2rb1y40.xyz
⚠️ Skipping file (suffix not allowed): p50_cc85slrsb2rb3y50.xyz
✅ Copied: P50_cc85sb2y30.xyz → cc85sb2y/
✅ Copied: P50_cc85y50.xyz → cc85y/
✅ Copied: P50_cc45y50.xyz → cc45y/
⚠️ Skipping file (suffix not allowed): p50_cc85blsb2rb3y40.xyz
⚠️ Skipping file (suffix not allowed): p50_cc85slrsb2y50.xyz
⚠️ Skipping file (suffix not allowed): p50_cc45ndsm2rb1y40.xyz
⚠️ Skipping file (suffix not allowed): p50_cc85qd20sb2rb3y40.xyz
✅ Copied: P50_cc45sm2y40.xyz → cc45sm2y/
⚠️ Skipping file (suffix not allowed): p50_cc45wdsm2rb1y40.xyz
⚠️ Skipping file (suffix not allowed): p50_cc85ndsb2rb3y40.xyz
✅ Copied: P50_cc85sb2y50.xyz → cc85sb2y/
✅ Copied: P50_cc45y30.xyz → cc45y/
⚠️ Skipping file (suffix not allowed): p50_cc85slry50.xyz
⚠️ Skipping fi

C:\Users\fuentesm\AppData\Local\Temp\ipykernel_26756\163341079.py:14: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  df = pd.read_csv(file_path, delim_whitespace=True, header=None, names=['x', 'y', 'z'])


Cloud Optimized GeoTIFF saved to N:\Deltabox\Postbox\Athanasiou, Panos\van_Sepehr\salinity_mekong\projections_gridded\cc45sm2rb1y\2030.tif
GeoJSON saved to N:\Deltabox\Postbox\Athanasiou, Panos\van_Sepehr\salinity_mekong\projections_gridded\cc45sm2rb1y\P50_cc45sm2rb1y30.geojson
